In [7]:
import os
from google.colab import userdata

In [8]:
os.environ['GOOGLE_API_KEY']=userdata.get('GOOGLE_API_KEY')

In [9]:
!pip install langchain chromadb openai tiktoken pypdf langchain_google-genai google-generativeai langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

In [5]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [6]:
docs = [doc1, doc2, doc3, doc4, doc5]

GOOGLE_API_KEY

In [13]:
from google.colab import userdata
api_key=userdata.get('GOOGLE_API_KEY')

In [23]:
import google.generativeai as genai

# Configure the API key
genai.configure(api_key=api_key)

print("Available Generative AI models:")
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(f"  - {m.name} (Embeddings)")

Available Generative AI models:
  - models/gemini-embedding-001 (Embeddings)
  - models/gemini-embedding-2-preview (Embeddings)
  - models/gemini-embedding-2 (Embeddings)


In [30]:
import os
import shutil

# Define the persist directory (not used for in-memory Chroma)
CHROMA_PERSIST_DIR = 'my_chroma_db'

# Delete the persist directory if it already exists to ensure a clean start
# This step is still good practice if you ever switch back to persistent mode
if os.path.exists(CHROMA_PERSIST_DIR):
    shutil.rmtree(CHROMA_PERSIST_DIR)
    print(f"Deleted existing directory: {CHROMA_PERSIST_DIR}")

# Initialize Chroma for in-memory operation
# Removed persist_directory argument to use in-memory Chroma
vector_store = Chroma(
    embedding_function=GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001', api_key=api_key),
    collection_name='sample'
)

print(f"Chroma vector store initialized as in-memory with model 'models/gemini-embedding-001'. Data will not be persisted.")

Chroma vector store initialized as in-memory with model 'models/gemini-embedding-001'. Data will not be persisted.


In [31]:
# add documents
print(f"Chroma vector store's embedding function model: {vector_store._embedding_function.model}")

# Attempt to add documents
vector_store.add_documents(docs)
print("Attempted to add documents.")

# Re-initialize vector_store to ensure it loads from the persisted directory
# This helps verify if add_documents successfully wrote to disk
reloaded_vector_store = Chroma(
    embedding_function=GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001', api_key=api_key),
    persist_directory=CHROMA_PERSIST_DIR,
    collection_name='sample'
)

# Verify if documents are present in the reloaded vector store
current_documents = reloaded_vector_store.get(include=['documents'])
if len(current_documents['documents']) > 0:
    print(f"Current documents in reloaded vector store: {len(current_documents['documents'])}.")
    if len(current_documents['documents']) == len(docs):
        print("All expected documents are present in the reloaded store.")
    else:
        print(f"Warning: Number of documents ({len(current_documents['documents'])}) does not match expected ({len(docs)}) in the reloaded store.")
else:
    print("No documents found in the reloaded vector store after addition attempt.")

Chroma vector store's embedding function model: models/gemini-embedding-001
Attempted to add documents.
No documents found in the reloaded vector store after addition attempt.


In [32]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['07091404-c07d-4a62-87c8-4e9afe817146',
  'bb2bf411-b735-4bca-85f3-c399e959ea1d',
  '71aa91f9-6f25-4e63-82df-291bb042f4be',
  'e6031c2b-3dcd-47c4-bf89-5b58380aa38a',
  '9368104c-abc8-4771-a0aa-6927bfdd5cd9'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [33]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [34]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6406819820404053),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.660536527633667)]

In [37]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="IPL players from Chennai Super Kings",
    filter={"team": "Chennai Super Kings"}
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.5662882924079895),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.573229193687439)]

In [38]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)


In [39]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['07091404-c07d-4a62-87c8-4e9afe817146',
  'bb2bf411-b735-4bca-85f3-c399e959ea1d',
  '71aa91f9-6f25-4e63-82df-291bb042f4be',
  'e6031c2b-3dcd-47c4-bf89-5b58380aa38a',
  '9368104c-abc8-4771-a0aa-6927bfdd5cd9'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [40]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [41]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['07091404-c07d-4a62-87c8-4e9afe817146',
  'bb2bf411-b735-4bca-85f3-c399e959ea1d',
  '71aa91f9-6f25-4e63-82df-291bb042f4be',
  'e6031c2b-3dcd-47c4-bf89-5b58380aa38a',
  '9368104c-abc8-4771-a0aa-6927bfdd5cd9'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

## Explanation of Functions and APIs

This section provides a detailed breakdown of the functions and APIs used in this notebook for building and interacting with a Chroma vector store.

### Core Python Libraries and Functions

*   **`os.environ.get(key)`**:
    *   **Purpose**: Retrieves the value of an environment variable named `key`. If the variable is not set, it returns `None`.
    *   **Usage in Notebook**: Used to attempt to get the `GOOGLE_API_KEY` from environment variables, although `userdata.get()` was ultimately used for Colab's secret manager.

*   **`os.path.exists(path)`**:
    *   **Purpose**: Checks if a `path` (file or directory) exists in the file system.
    *   **Usage in Notebook**: Used to check if the `CHROMA_PERSIST_DIR` (the directory for persistent Chroma storage) exists before attempting to delete it.

*   **`shutil.rmtree(path)`**:
    *   **Purpose**: Recursively deletes a directory and its contents at the given `path`.
    *   **Usage in Notebook**: Used to ensure a clean slate for the Chroma vector store by removing any previous persistent data before initialization.

*   **`google.colab.userdata.get(key)`**:
    *   **Purpose**: A Colab-specific function to securely retrieve secrets stored in the Colab secrets manager using their `key`.
    *   **Usage in Notebook**: Used to retrieve the `GOOGLE_API_KEY` from the Colab environment securely.

### Google Generative AI API (`google.generativeai`)

*   **`genai.configure(api_key)`**:
    *   **Purpose**: Initializes the Google Generative AI client with the provided `api_key`.
    *   **Usage in Notebook**: Sets up the API key for authentication with the Generative AI services.

*   **`genai.list_models()`**:
    *   **Purpose**: Lists all available models in the Google Generative AI API.
    *   **Usage in Notebook**: Used to identify suitable embedding models by filtering for those that support `embedContent` generation methods, which helped in discovering `models/gemini-embedding-001`.

### LangChain Library

#### `langchain_core.documents.Document`

*   **`Document(page_content, metadata)`**:
    *   **Purpose**: A fundamental data structure in LangChain representing a piece of text content along with optional metadata.
    *   **Parameters**:
        *   `page_content` (str): The main text of the document.
        *   `metadata` (dict, optional): A dictionary of arbitrary key-value pairs associated with the document.
    *   **Usage in Notebook**: Documents are created for each IPL player, storing their description in `page_content` and their team in `metadata`.

#### `langchain_google_genai.GoogleGenerativeAIEmbeddings`

*   **`GoogleGenerativeAIEmbeddings(model, api_key)`**:
    *   **Purpose**: An embedding class that interfaces with Google's Generative AI models to convert text into numerical vector representations (embeddings).
    *   **Parameters**:
        *   `model` (str): The name of the embedding model to use (e.g., `'models/gemini-embedding-001'`)
        *   `api_key` (str): Your Google API key for authentication.
    *   **Usage in Notebook**: This instance is passed to Chroma to specify how the text documents should be converted into embeddings before being stored in the vector database.

#### `langchain_community.vectorstores.Chroma`

*   **`Chroma(embedding_function, collection_name, persist_directory=None)`** (Constructor):
    *   **Purpose**: Initializes a Chroma vector store. It can be initialized as in-memory or persistent.
    *   **Parameters**:
        *   `embedding_function`: An instance of an embedding class (like `GoogleGenerativeAIEmbeddings`) that Chroma uses to generate embeddings for documents.
        *   `collection_name` (str): A name for the collection within the Chroma database.
        *   `persist_directory` (str, optional): If provided, Chroma will store its data persistently in this directory. If `None`, Chroma operates in-memory.
    *   **Usage in Notebook**: `vector_store` is initialized as an in-memory Chroma instance, using `GoogleGenerativeAIEmbeddings` for vectorization.

*   **`vector_store.add_documents(documents)`**:
    *   **Purpose**: Adds a list of `Document` objects to the vector store.
    *   **Parameters**:
        *   `documents` (list of `Document`): The documents to be added.
    *   **Usage in Notebook**: Used to ingest the created IPL player documents into the Chroma vector store.

*   **`vector_store.get(include=['embeddings', 'documents', 'metadatas'])`**:
    *   **Purpose**: Retrieves documents, their embeddings, and metadata from the vector store.
    *   **Parameters**:
        *   `include` (list of str): A list specifying which components to retrieve (e.g., `'documents'`, `'embeddings'`, `'metadatas'`, `'ids'`, `'uris'`).
    *   **Usage in Notebook**: Used to inspect the contents of the vector store, confirming which documents were added and what their associated embeddings and metadata are.

*   **`vector_store.similarity_search(query, k)`**:
    *   **Purpose**: Performs a similarity search, finding the `k` most semantically similar documents to a given `query` string.
    *   **Parameters**:
        *   `query` (str): The text query to search for.
        *   `k` (int): The number of top similar documents to retrieve.
    *   **Usage in Notebook**: Used to find IPL players most similar to the query "Who among these are a bowler?".

*   **`vector_store.similarity_search_with_score(query, k, filter)`**:
    *   **Purpose**: Similar to `similarity_search`, but also returns a similarity score for each retrieved document, indicating its relevance to the query. It also allows for metadata filtering.
    *   **Parameters**:
        *   `query` (str): The text query.
        *   `k` (int): The number of top similar documents to retrieve.
        *   `filter` (dict, optional): A dictionary to filter documents based on their metadata (e.g., `{"team": "Chennai Super Kings"}`).
    *   **Usage in Notebook**: Used to retrieve documents with scores and to filter for specific teams, demonstrating more advanced search capabilities.

*   **`vector_store.update_document(document_id, document)`**:
    *   **Purpose**: Updates an existing document in the vector store identified by its `document_id` with new `Document` content.
    *   **Parameters**:
        *   `document_id` (str): The unique ID of the document to update.
        *   `document` (`Document`): The new `Document` object to replace the old one.
    *   **Usage in Notebook**: Used to modify the details of an existing IPL player document (Virat Kohli in this case).

*   **`vector_store.delete(ids)`**:
    *   **Purpose**: Deletes documents from the vector store based on a list of their unique IDs.
    *   **Parameters**:
        *   `ids` (list of str): A list of document IDs to delete.
    *   **Usage in Notebook**: Used to remove a specific IPL player document from the store.

## Comprehensive Explanation of Functions and APIs

This section offers a comprehensive and in-depth explanation of every function, class, and API used throughout this notebook. Understanding these components is crucial for effectively working with LangChain, Google Generative AI, and vector databases like Chroma.

### Core Python Libraries and Functions

These are standard Python functionalities often used for system-level operations or interacting with the environment.

*   **`os.environ.get(key)`**
    *   **Purpose**: This function from the `os` module (operating system module) is used to retrieve the value of an environment variable. Environment variables are dynamic named values that can affect the way running processes behave. They are often used for configuration settings, especially for sensitive information like API keys.
    *   **Parameters**:
        *   `key` (str): The name of the environment variable you want to retrieve.
        *   `default` (optional): A value to return if the environment variable is not found. If omitted, `None` is returned.
    *   **Usage in Notebook**: Initially, we might attempt to fetch `GOOGLE_API_KEY` using `os.environ.get('GOOGLE_API_KEY')`. However, in cloud environments like Google Colab, directly setting environment variables for secrets can be less secure or persistent. This is why `google.colab.userdata.get()` is preferred for Colab notebooks.
    *   **Best Practice**: For local development, `os.environ` is common. For cloud platforms, use platform-specific secret management (like Colab's `userdata`).

*   **`os.path.exists(path)`**
    *   **Purpose**: A utility function from the `os.path` submodule that checks if a specified file or directory path actually exists on the filesystem.
    *   **Parameters**:
        *   `path` (str): The file or directory path to check.
    *   **Returns**: `True` if the path exists, `False` otherwise.
    *   **Usage in Notebook**: Used to determine if the `CHROMA_PERSIST_DIR` (e.g., `'my_chroma_db'`) exists before attempting to delete it. This prevents errors if you run the notebook multiple times and the directory hasn't been created yet.
    *   **Example**: `if os.path.exists('my_folder'): print('Folder exists!')`

*   **`shutil.rmtree(path)`**
    *   **Purpose**: This function from the `shutil` module (shell utilities) is used for high-level file operations. `rmtree` stands for 'remove tree' and recursively deletes a directory and all its contents.
    *   **Parameters**:
        *   `path` (str): The path to the directory you want to delete.
    *   **Caution**: Use with extreme care, as it permanently deletes files and directories without confirmation.
    *   **Usage in Notebook**: Employed to ensure a clean state for the Chroma vector store. By deleting `CHROMA_PERSIST_DIR` before initializing Chroma, we guarantee that no old, potentially corrupt, or incompatible data interferes with the new setup. This was particularly useful when debugging persistent Chroma issues.

*   **`google.colab.userdata.get(key)`**
    *   **Purpose**: This Colab-specific function provides a secure way to access secrets (like API keys) stored in Google Colab's Secrets Manager. Secrets are stored separately from your notebook and are not exposed in the notebook's code or output, enhancing security.
    *   **Parameters**:
        *   `key` (str): The name of the secret as defined in the Colab Secrets Manager.
    *   **Returns**: The value of the secret.
    *   **Usage in Notebook**: This is the recommended method for retrieving your `GOOGLE_API_KEY` within a Colab environment, providing a safer alternative to hardcoding or direct environment variables for sensitive data.
    *   **Benefit**: Prevents accidental exposure of API keys if you share your notebook.

### Google Generative AI API (`google.generativeai`)

This library allows Python applications to interact with Google's powerful Generative AI models, including text generation, embeddings, and more.

*   **`genai.configure(api_key)`**
    *   **Purpose**: This is the primary function to initialize and configure the Google Generative AI client. It sets up the authentication credentials that your application will use to make API calls to Google's AI services.
    *   **Parameters**:
        *   `api_key` (str): Your Google API key, obtained from Google AI Studio.
    *   **Usage in Notebook**: Once `api_key` is retrieved (e.g., from `userdata.get()`), `genai.configure(api_key=api_key)` must be called *before* attempting to interact with any Generative AI models. Without this, API calls will fail due to authentication errors.

*   **`genai.list_models()`**
    *   **Purpose**: This function retrieves a list of all Generative AI models that are currently available through your configured API key. This is invaluable for discovering model names, capabilities, and ensuring you are using a supported model for your specific task.
    *   **Returns**: An iterable (generator) of `Model` objects, each containing information like `name`, `supported_generation_methods`, and `version`.
    *   **Usage in Notebook**: We used `genai.list_models()` and filtered its output to find models that support the `embedContent` generation method. This diagnostic step was crucial for identifying the correct embedding model name, `models/gemini-embedding-001`, after previous attempts with incorrect model names led to errors.
    *   **Benefit**: Helps avoid `GoogleGenerativeAIError`s due to incorrect or unsupported model names.

### LangChain Library

LangChain is a framework designed to simplify the creation of applications powered by large language models (LLMs). It provides components for various LLM-related tasks, including document loading, splitting, embeddings, vector stores, and agent orchestration.

#### `langchain_core.documents.Document`

This is a foundational data structure within LangChain, representing a piece of information or content.

*   **`Document(page_content, metadata)`**
    *   **Purpose**: The `Document` class serves as a standardized way to encapsulate text content along with associated metadata. This structure is central to how LangChain handles data that will be processed by LLMs, especially when using vector stores or retrieval augmented generation (RAG).
    *   **Parameters**:
        *   `page_content` (str): This is the primary textual content of the document. It holds the actual information you want to store, search, or process (e.g., a paragraph from an article, a sentence, or a player's description).
        *   `metadata` (dict, optional): A dictionary of key-value pairs that describe the document. This is extremely useful for adding contextual information without cluttering the `page_content`. Metadata can be used for filtering, categorizing, or providing additional context during retrieval.
    *   **Usage in Notebook**: We created several `Document` objects, each representing an IPL player. The `page_content` contains the player's description, and the `metadata` includes their `team`. This `metadata` was later used for filtering search results.
    *   **Importance**: Standardizing data into `Document` objects allows various LangChain components (loaders, splitters, embedders, vector stores, retrievers) to seamlessly work together.

#### `langchain_google_genai.GoogleGenerativeAIEmbeddings`

This class provides the interface to generate embeddings using Google's Generative AI models within the LangChain framework.

*   **`GoogleGenerativeAIEmbeddings(model, api_key)`**
    *   **Purpose**: An embedding model converts text (like a document or a query) into a dense numerical vector (an embedding). These vectors capture the semantic meaning of the text, allowing for operations like similarity search. This class specifically leverages Google's Generative AI embedding models.
    *   **Parameters**:
        *   `model` (str): Specifies which Google Generative AI embedding model to use (e.g., `'models/gemini-embedding-001'`). The choice of model impacts the quality and dimensionality of the embeddings.
        *   `api_key` (str): Your Google API key, required for authenticating with the Google Generative AI service.
    *   **How it works**: When `GoogleGenerativeAIEmbeddings` processes text, it sends the text to Google's API, which then returns a vector representation. This vector is then used by the vector store for indexing and similarity comparisons.
    *   **Usage in Notebook**: An instance of this class was created and passed to the `Chroma` vector store. This tells Chroma to use `models/gemini-embedding-001` (with our `api_key`) to generate embeddings for all documents added to the store and for any subsequent queries.
    *   **Key Concept (Embeddings)**: Texts that are semantically similar will have embedding vectors that are 'close' to each other in the high-dimensional space. Vector stores use distance metrics (like cosine similarity) to find these 'closest' vectors.

#### `langchain_community.vectorstores.Chroma`

Chroma is an open-source vector database. LangChain's `Chroma` integration allows you to easily store, manage, and query document embeddings.

*   **`Chroma(embedding_function, collection_name, persist_directory=None)`** (Constructor)
    *   **Purpose**: This is the constructor for the `Chroma` vector store. It initializes a new Chroma instance, setting up how documents will be embedded and where the data will be stored.
    *   **Parameters**:
        *   `embedding_function`: **Crucial parameter**. This is an instance of an embedding class (like `GoogleGenerativeAIEmbeddings`) that Chroma will use internally. Every document added to Chroma, and every query made, will be converted into an embedding using this function.
        *   `collection_name` (str): A logical name for the set of documents you're storing. A single Chroma database can contain multiple collections, each with its own set of documents and embeddings.
        *   `persist_directory` (str, optional): This parameter determines whether the Chroma database will be **persistent** (saved to disk) or **in-memory**. If a string path is provided (e.g., `'./my_chroma_db'`), Chroma will save its data to that directory, allowing it to be reloaded across different Python sessions. If `None` (as in our final fix), the data is stored only in RAM and will be lost when the Python process ends.
    *   **Usage in Notebook**: Our `vector_store` was initialized as an **in-memory** Chroma instance. This decision was made to circumvent the `InternalError: attempt to write a readonly database` error encountered when trying to use persistent storage in the Colab environment.
    *   **Trade-off (In-memory vs. Persistent)**: In-memory is fast and avoids disk issues but loses data. Persistent stores data but requires write access and can be slower for very large datasets.

*   **`vector_store.add_documents(documents)`**
    *   **Purpose**: This method is used to add one or more LangChain `Document` objects to the Chroma vector store. When you call this method, Chroma uses the `embedding_function` you provided during initialization to convert the `page_content` of each `Document` into an embedding vector.
    *   **Parameters**:
        *   `documents` (list of `Document`): A list containing the `Document` objects you wish to add to the vector store.
    *   **How it works**: For each document, its text content is embedded, and both the embedding and the original document (including its metadata) are stored in the Chroma collection.
    *   **Usage in Notebook**: `vector_store.add_documents(docs)` was used to ingest our list of IPL player documents into the Chroma database. After this, these documents become searchable based on their semantic content.

*   **`vector_store.get(include=['embeddings', 'documents', 'metadatas'])`**
    *   **Purpose**: This method allows you to retrieve data directly from the Chroma collection. It's useful for inspecting what's currently stored in your vector database, debugging, or performing operations on the raw data.
    *   **Parameters**:
        *   `ids` (list of str, optional): If provided, retrieves only the documents with these specific IDs.
        *   `where` (dict, optional): A dictionary for metadata filtering (e.g., `{'team': 'Mumbai Indians'}`). This works like an `AND` clause.
        *   `where_document` (dict, optional): Filters based on the content of the documents.
        *   `limit` (int, optional): Maximum number of results to return.
        *   `offset` (int, optional): Skips a specified number of results.
        *   `include` (list of str, optional): **Crucial parameter**. This specifies which components of the stored data you want to retrieve. Common options include:
            *   `'embeddings'`: The numerical vectors generated from the document content.
            *   `'documents'`: The original `page_content` strings.
            *   `'metadatas'`: The associated metadata dictionaries.
            *   `'ids'`: The unique IDs assigned to each document by Chroma.
    *   **Returns**: A dictionary containing the requested components (e.g., `{'ids': [...], 'documents': [...], 'metadatas': [...]}`).
    *   **Usage in Notebook**: `vector_store.get(include=['embeddings', 'documents', 'metadatas'])` was used multiple times to verify that documents were added, updated, and deleted correctly, and to examine their associated embeddings and metadata.

*   **`vector_store.similarity_search(query, k)`**
    *   **Purpose**: This is one of the primary methods for querying the vector store. It finds the `k` documents in the collection that are most semantically similar to your `query` string.
    *   **Parameters**:
        *   `query` (str): The text string representing your search query.
        *   `k` (int, optional): The number of most similar documents to retrieve. Defaults to 4.
        *   `filter` (dict, optional): Allows you to apply metadata filters (similar to `where` in `get`).
    *   **How it works**: When you call `similarity_search`, the `query` string is first converted into an embedding vector using the `embedding_function` defined during Chroma initialization. Then, Chroma compares this query embedding to all the document embeddings stored in the collection using a distance metric (e.g., cosine similarity). The `k` documents with the 'closest' (most similar) embeddings are returned.
    *   **Returns**: A list of `Document` objects, ordered by similarity (most similar first).
    *   **Usage in Notebook**: `vector_store.similarity_search(query='Who among these are a bowler?', k=2)` was used to find the two IPL players whose descriptions were most relevant to the concept of a 'bowler'.

*   **`vector_store.similarity_search_with_score(query, k, filter)`**
    *   **Purpose**: This method is an extension of `similarity_search`. It performs the same semantic search but also returns a similarity score for each retrieved document. Additionally, it explicitly supports metadata `filter`ing.
    *   **Parameters**:
        *   `query` (str): The text string for your search.
        *   `k` (int, optional): The number of top similar documents to retrieve.
        *   `filter` (dict, optional): A dictionary to apply metadata filtering. Only documents that match this filter will be considered for the similarity search. For example, `{'team': 'Chennai Super Kings'}`.
    *   **Returns**: A list of tuples, where each tuple contains `(Document, score)`. The `score` indicates the degree of similarity, with lower scores often indicating higher similarity (depending on the distance metric used by the embedding model and vector store, which typically use cosine distance where 0 is identical and 1 is completely dissimilar, or cosine similarity where 1 is identical and -1 is completely dissimilar).
    *   **Usage in Notebook**: `vector_store.similarity_search_with_score(query='IPL players from Chennai Super Kings', filter={'team': 'Chennai Super Kings'})` was used to demonstrate filtering documents by a specific team while also getting the similarity score to the query.
    *   **Score Interpretation**: The exact meaning and range of the score depend on the underlying embedding model and the distance metric. Typically, a lower score (closer to 0) in cosine distance indicates higher similarity.

*   **`vector_store.update_document(document_id, document)`**
    *   **Purpose**: This method allows you to modify an existing document in the vector store. You provide the unique ID of the document you want to change and the new `Document` object to replace it.
    *   **Parameters**:
        *   `document_id` (str): The unique ID of the document to be updated. You can find these IDs using `vector_store.get(include=['ids'])`.
        *   `document` (`Document`): The new `Document` object containing the updated `page_content` and `metadata`. The `embedding_function` will be used to re-embed the new content.
    *   **Usage in Notebook**: `vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)` was used to revise the description of a specific IPL player (Virat Kohli) by replacing his original document with an `updated_doc1`.
    *   **Note**: Updating a document usually means deleting the old entry and adding a new one with the same ID, effectively replacing its embedding and content.

*   **`vector_store.delete(ids)`**
    *   **Purpose**: This method allows you to remove specific documents from the Chroma vector store based on their unique IDs.
    *   **Parameters**:
        *   `ids` (list of str): A list of one or more unique document IDs to be deleted from the collection.
    *   **Usage in Notebook**: `vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])` was used to remove the document corresponding to Virat Kohli (or whichever document had that specific ID) from the vector store.
    *   **Effect**: Once deleted, the document and its embedding will no longer be retrieved in similarity searches.